In [ ]:
!pip install -q gradio wikipedia-api sentence-transformers transformers torch


In [ ]:
import pandas as pd

data = {
    "question": [
        "Who is the founder of Pakistan?",
        "What are the colors in Pakistan flag?",
        "What is the capital of Pakistan?",
        "How many provinces are there in Pakistan?",
        "What is the national dish of Pakistan?"
    ],
    "answer": [
        "The founder of Pakistan is Muhammad Ali Jinnah.",
        "The flag of Pakistan is green and white, with a white crescent moon and a five-pointed star.",
        "The capital of Pakistan is Islamabad.",
        "Pakistan has four provinces: Punjab, Sindh, Khyber Pakhtunkhwa, and Balochistan.",
        "The national dish of Pakistan is Nihari."
    ]
}

faqs = pd.DataFrame(data)
faqs


,question,answer
0,Who is the founder of Pakistan?,The founder of Pakistan is Muhammad Ali Jinnah.
1,What are the colors in Pakistan flag?,"The flag of Pakistan is green and white, with ..."
2,What is the capital of Pakistan?,The capital of Pakistan is Islamabad.
3,How many provinces are there in Pakistan?,"Pakistan has four provinces: Punjab, Sindh, Kh..."
4,What is the national dish of Pakistan?,The national dish of Pakistan is Nihari.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sbert = SentenceTransformer("all-MiniLM-L6-v2")

faq_questions = faqs["question"].tolist()
faq_emb = sbert.encode(faq_questions, convert_to_numpy=True)
print("✅ FAQ embeddings shape:", faq_emb.shape)


✅ FAQ embeddings shape: (5, 384)


In [ ]:
def faq_match(query):
    """Finds the most similar FAQ and returns (best_idx, best_score)."""
    q_emb = sbert.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, faq_emb)[0]
    best_idx = int(np.argmax(sims))
    best_score = float(sims[best_idx])
    return best_idx, best_score


In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent="ColabHybridBot/1.0", language="en")

def wiki_search_and_context(query, max_chars=500):
    page = wiki.page(query)
    if page.exists():
        return page.summary[:max_chars]
    else:
        return None


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
llm = pipeline("text2text-generation", model=model, tokenizer=tokenizer)
print("✅ Loaded:", model_name)


Device set to use cpu


✅ Loaded: google/flan-t5-base


In [ ]:
CONF_THRESH = 0.65  # how confident to trust FAQ over LLM

def hybrid_answer(query):
    idx, score = faq_match(query)
    if score >= CONF_THRESH:
        return f"[FAQ] {faqs.iloc[idx]['answer']}"

    # Otherwise Wikipedia + LLM
    context = wiki_search_and_context(query)
    if context:
        prompt = f"Answer the question using this context:\n{context}\n\nQuestion: {query}"
    else:
        prompt = query

    output = llm(prompt, max_length=150, temperature=0.7)
    return output[0]["generated_text"]


In [ ]:
import gradio as gr

def chat_interface(query):
    return hybrid_answer(query)

demo = gr.Interface(fn=chat_interface, inputs="text", outputs="text",
                    title="💬 Hybrid FAQ + Wikipedia + Flan-T5 Chatbot",
                    description="Answers from FAQ if possible, otherwise searches Wikipedia and generates an intelligent response.")
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://843fe0ad81bda41ebe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def chat_fn(message, history):
    if history is None:
        history = []
    try:
        res = hybrid_answer(message)
        label = f"[{res['source']}]"
        reply = f"{res['answer']}"
    except Exception as e:
        label = "[error]"
        reply = f"Error: {str(e)}"
    history.append((message, f"{label} {reply}"))
    return history, history

with gr.Blocks() as demo:
    gr.Markdown("## Hybrid FAQ + Wikipedia-RAG + Flan-T5 (Colab)\n*FAQ prioritized; otherwise Wikipedia retrieval + Flan-T5 generation.*")
    chatbot = gr.Chatbot()
    txt = gr.Textbox(placeholder="Ask any question...", lines=2)
    txt.submit(chat_fn, [txt, chatbot], [chatbot, chatbot])
    gr.Button("Clear").click(lambda: None, None, chatbot)
demo.launch(share=True)


/tmp/ipython-input-1399158308.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d88e8f336dfb07a4e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
